<a href="https://colab.research.google.com/github/Bavesh-08/building-CNN-with-ResNet-Transfer-Model-/blob/main/building_resnet_model_CIFAE_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.1 MB/s eta 0:00:00


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import keras_tuner as kt

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

num_classes = 10
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test  = tf.keras.utils.to_categorical(y_test,  num_classes)

print(x_train.shape, y_train.shape)  # (50000, 32, 32, 3) (50000, 10)
print(x_test.shape, y_test.shape)    # (10000, 32, 32, 3) (10000, 10)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step
(50000, 32, 32, 3) (50000, 10)
(10000, 32, 32, 3) (10000, 10)


In [6]:
def build_model(hp):
    inputs = layers.Input(shape=(32, 32, 3))

    # Augmentation (runs only during training)
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(hp.Float("rotation", 0.05, 0.30, step=0.05))(x)
    x = layers.RandomZoom(hp.Float("zoom", 0.00, 0.20, step=0.05))(x)

    # Upsample to match ImageNet backbone expectations
    img_size = hp.Choice("img_size", [96, 128, 160])
    x = layers.Resizing(img_size, img_size)(x)

    # Preprocess for ResNetV2
    x = tf.keras.applications.resnet_v2.preprocess_input(x)

    # Backbone
    base = tf.keras.applications.ResNet50V2(
        include_top=False,
        weights="imagenet",
        input_tensor=x
    )
    base.trainable = True  # freeze pretrained backbone

    # Head
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)

    units = hp.Int("units", min_value=128, max_value=512, step=128)
    x = layers.Dense(units, activation="relu")(x)

    drop = hp.Float("dropout", min_value=0.2, max_value=0.6, step=0.1)
    x = layers.Dropout(drop)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inputs, outputs)

    lr = hp.Choice("lr", [1e-3, 3e-4, 1e-4])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [7]:
tuner = kt.BayesianOptimization(
    build_model,
    objective="val_accuracy",
    max_trials=2,
    directory="cifar10_tf_tuning",
    project_name="resnet50v2_pretrained"
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2),
]

tuner.search(
    x_train, y_train,
    epochs=5,
    validation_split=0.1,   # uses all 50k, splits 45k/5k internally
    callbacks=callbacks,
    batch_size=64
)

Trial 2 Complete [00h 28m 03s]
val_accuracy: 0.9330000281333923

Best val_accuracy So Far: 0.9330000281333923
Total elapsed time: 00h 41m 05s


In [8]:
best_model = tuner.get_best_models(1)[0]
test_loss, test_acc = best_model.evaluate(x_test, y_test, verbose=0)
print("Test accuracy:", test_acc)

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 358 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Test accuracy: 0.9269000291824341
